# Lọc mấy tấm hình lấy ra train dựa theo dữ liệu đã filter 5 core
File df_inter.label.parquet, df_inter.parquet lấy ở bước 1 hoặc 2 điều được

In [1]:
import os
import pandas as pd

In [2]:
PATH = "../data/2023/"

In [3]:
os.makedirs(os.path.join(PATH, "step0_clean_data"), exist_ok=True)

In [5]:
df = pd.read_parquet(os.path.join(PATH, "step0_clean_data", "df_meta.parquet"))

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35997 entries, 0 to 35996
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    35236 non-null  str    
 1   title            35997 non-null  str    
 2   average_rating   35997 non-null  float64
 3   rating_number    35997 non-null  int64  
 4   features         35997 non-null  object 
 5   description      35997 non-null  object 
 6   price            17404 non-null  float64
 7   images           35997 non-null  object 
 8   videos           35997 non-null  object 
 9   store            35849 non-null  str    
 10  categories       35997 non-null  object 
 11  details          35997 non-null  object 
 12  asin             35997 non-null  str    
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
dtypes: float64(4), int64(1), object(7), str(4)
memory usage: 9.1+ MB


In [8]:
df.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,asin,bought_together,subtitle,author
0,Baby,"Chicco Viaro Travel System, Teak",4.6,125,"[Aluminum, Imported, Convenient one-hand quick...","[Product Description, For ultimate convenience...",NaN,[{'hi_res': 'https://m.media-amazon.com/images...,"[{'title': 'Viaro Demo Video', 'url': 'https:/...",Chicco,"[Baby Products, Strollers & Accessories, Strol...","{'': None, 'ABPA Partslink Number': None, 'AC ...",B01C4319LO,NaN,NaN,None
1,Baby,Nuby iMonster Toddler Bowl,4.4,52,[Makes feeding fun for baby and easier for par...,[When babies begin to show interest in feeding...,NaN,[{'hi_res': 'https://m.media-amazon.com/images...,[],Nuby,"[Baby Products, Feeding, Solid Feeding, Dishes]","{'': None, 'ABPA Partslink Number': None, 'AC ...",B0083SXABC,NaN,NaN,None
2,Baby,XMWEALTHY Unisex Infant Swaddle Blankets Soft ...,4.7,5919,"[Outside 100% Acrylic, Inside 100%polyester, 👶...",[],28.99,[{'hi_res': 'https://m.media-amazon.com/images...,[{'title': 'Perfect Winter time Baby Burrito. ...,XMWEALTHY,"[Baby Products, Nursery, Bedding, Blankets & S...","{'': None, 'ABPA Partslink Number': None, 'AC ...",B07JM4RK9T,NaN,NaN,None
3,Baby,Cloele Kids Toddler Pillowcase Envelope Travel...,4.6,1169,"[100% Cotton, 1.Toddler pillowcase size: 1 pac...",[],5.99,[{'hi_res': 'https://m.media-amazon.com/images...,"[{'title': 'EVERYDAY KIDS 2 Pack Pillowcases',...",Cloele,"[Baby Products, Nursery, Bedding, Toddler Bedd...","{'': None, 'ABPA Partslink Number': None, 'AC ...",B08F1VWF5P,NaN,NaN,None
4,"Arts, Crafts & Sewing",BERON 16 Pieces 4 Inches Colorful Handmade Chi...,4.7,206,"[Made of chiffon fabric, soft and comfortable,...",[Colorful Chiffon Flowers Make Your Life Brigh...,11.99,"[{'hi_res': None, 'large': 'https://m.media-am...",[{'title': 'Chiffon Flower Hair Bows Fully Lin...,BERON,"[Baby Products, Baby Care, Hair Care, Hair Acc...","{'': None, 'ABPA Partslink Number': None, 'AC ...",B01DDDXTA8,NaN,NaN,None


In [ ]:
# preprocessing\data\2023\step0_clean_data

In [12]:
import shutil
from pathlib import Path

# Thay bằng thư mục chứa ảnh gốc của bạn
image_src_dir = Path(PATH) / "step0_clean_data" / "Baby_Products"
# Thư mục đích để lưu ảnh đã lọc
image_dst_dir = Path(PATH) / "step0_clean_data" / "filtered_images"
image_dst_dir.mkdir(parents=True, exist_ok=True)

if "asin" not in df.columns:
    raise ValueError("Cột 'asin' không tồn tại trong df")

asin_set = set(df["asin"].dropna().astype(str).str.strip())
found_asins = set()
copied = 0
skipped = 0
for src_path in image_src_dir.rglob("*"):
    if not src_path.is_file():
        continue
    asin = src_path.stem
    if asin in asin_set:
        found_asins.add(asin)
        dst_path = image_dst_dir / src_path.name
        if dst_path.exists():
            skipped += 1
            continue
        shutil.copy2(src_path, dst_path)
        copied += 1

# Đánh dấu rows có file ảnh
df["has_image"] = df["asin"].astype(str).str.strip().isin(found_asins)
if df["asin"].isna().any():
    df.loc[df["asin"].isna(), "has_image"] = False

missing_asins = sorted(asin_set - found_asins)
print(f"✅ Đã copy {copied} ảnh có ASIN trong df vào {image_dst_dir} (đã bỏ qua {skipped} file đã tồn tại)")
print(f"ℹ️ Tổng ASIN trong df: {len(asin_set)}, tìm thấy file: {len(found_asins)}, thiếu file: {len(missing_asins)}")
if missing_asins:
    print("Các ASIN có trong df nhưng không tìm thấy file:")
    print(missing_asins[:50])
    if len(missing_asins) > 50:
        print(f"...và còn {len(missing_asins) - 50} ASIN nữa")

✅ Đã copy 0 ảnh có ASIN trong df vào ..\data\2023\step0_clean_data\filtered_images (đã bỏ qua 35960 file đã tồn tại)
ℹ️ Tổng ASIN trong df: 35997, tìm thấy file: 35960, thiếu file: 37
Các ASIN có trong df nhưng không tìm thấy file:
['B00005QI1H', 'B000GECKOY', 'B0017WEH1S', 'B0018CJ7G2', 'B00192H1KA', 'B00192LH3C', 'B001R1PJY8', 'B004AAIQ1G', 'B0092KKTQ4', 'B009WUPGQC', 'B00BX8RRJA', 'B00DGN4R1Q', 'B00OPZR8L0', 'B01E5A2T72', 'B073N2GY2G', 'B0754V1ZRB', 'B07634N152', 'B079659G3W', 'B07DGJ65YK', 'B07RNNR4VH', 'B07T94P87H', 'B07TLHM5S8', 'B085QPYTJ8', 'B086GVZLPV', 'B08GKLXHYD', 'B08XYQL2VF', 'B099J4VRS2', 'B09HQ8X7FB', 'B09JP1HXN8', 'B09JP21WCK', 'B09JVR2566', 'B09KRQ18Z1', 'B09MGLQ35Y', 'B09TZSHWXD', 'B0B4JNTNX1', 'B0BFYZ13CJ', 'B0C6WJ46L6']


In [11]:
df[df["asin"].isna()]

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,asin,bought_together,subtitle,author


In [14]:
df[df["asin"].isin(missing_asins)].shape

(37, 16)

In [15]:
df[df["asin"].isin(asin_set)].shape

(35997, 16)

In [16]:
df[df["asin"].isin(found_asins)].shape

(35960, 16)

In [20]:
df["has_image"] = df["asin"].astype(str).str.strip().isin(found_asins)

In [21]:
df["has_image"].value_counts()

has_image
True     35960
False       37
Name: count, dtype: int64

In [26]:
# df[df["has_image"] == False]["images"].to_excel(os.path.join(PATH, "step0_clean_data", "missing_asins.xlsx"), index=False)

In [22]:
df.to_parquet(os.path.join(PATH, "step0_clean_data", "df_meta.with_image.parquet"), index=False)

In [28]:
df_inter = pd.read_parquet(os.path.join(PATH, "step1_rating_to_inter", "df_inter.parquet"))

In [29]:
df_dont_have_image = df[df["has_image"] == False]

In [31]:
df_inter.shape

(1240219, 6)

In [36]:
# 707/1240219*100

In [30]:
df_inter[df_inter["asin"].isin(df_dont_have_image["asin"])]

,userID,itemID,rating,timestamp,reviewerID,asin
1919,90,1660,1,1273002200000,AGYAEIYXACDC4DFXFBVL544OEP3Q,B0017WEH1S
2137,104,1830,5,1621562011439,AGZMKHWSCB3UXDGFUPFRZSL4EAWQ,B085QPYTJ8
2212,104,1894,5,1578117324530,AGZMKHWSCB3UXDGFUPFRZSL4EAWQ,B07RNNR4VH
2215,105,1897,5,1650125244565,AFNL6B7KCS3NN3K5ADRPXO7EINUQ,B09KRQ18Z1
3265,157,1897,5,1651608501638,AFLBMDZDIPCHMWWA7CXQXITPBJKA,B09KRQ18Z1
...,...,...,...,...,...,...
1235386,149804,7316,5,1615897849836,AGCK4GIR7U7NS4VERJNV7BARRIEA,B0BFYZ13CJ
1237548,150197,12972,5,1404154597000,AHA6NZCBP4PF5XHJS5SDCMHDPITQ,B00DGN4R1Q
1238639,150391,35855,5,1132760793000,AGTXN5FXDYRI3ZXTSZFO4W45KYNA,B00005QI1H
1240062,150644,35855,5,1027971719000,AFCJ62KUNUW7NI6UIWPNLVC4UVCA,B00005QI1H
